# A.5 答錯是沒找到還是沒用好？


三家書店只答對一家，可能是沒有找到公告，也可能找到後讀錯。先為每題列出所需來源，再保存實際取回的編號與答案，就能把兩站責任分開。

下面是三份人工診斷紀錄。hit表示所需來源出現在取回清單，correct表示答案符合固定真值。甲兩項都對，乙找到但答錯，丙既沒找到也沒答對。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
examples = [
    {"question": "甲店地址", "hit": True, "correct": True},
    {"question": "乙店地址", "hit": True, "correct": False},
    {"question": "丙店地址", "hit": False, "correct": False},
]
count = len(examples)
recall = sum(e["hit"] for e in examples) / count
accuracy = sum(e["correct"] for e in examples) / count
print("有正確來源", recall)
print("答案正確", accuracy)
for e in examples:
    if not e["correct"]:
        print(e["question"], "已找到但答錯" if e["hit"] else "尚未找到所需來源")

有正確來源2/3，答案正確1/3。這裡每題只有一份指定來源，來源命中分母也是題數；需要多份來源的題目要另說「至少找到一份」還是「全部找到」。答案判準也預先固定，不能只看回答流暢。

先處理乙：核對最終提示是否完整保留公告，再查地址與引用。丙先查文件、查詢和排序。可以另把正確公告直接提供給失敗題，固定模型重新回答；若這時答對，檢索很可能是瓶頸，但提示長度或其他資料也變時要記下，不能當絕對因果證明。

只把乙的correct改True，來源比例仍2/3，答案變2/3。這個紙上變化發生在使用資料，不是多找了一份文件。後面的長上下文與對話題也沿這個區分保存中途資料。

<details>
<summary>補充：舊12題RAG的分項矩陣</summary>

正式組固定同一12個新店名，每種條件都實際生成完整12題，沒有跳過失敗題。下表的「按協議答對」在缺所問公告時期待`UNKNOWN`；「地址與引用答對」才要求新地址與指定來源相符，兩欄不能混算。

| 當次上下文 | 按協議答對 | 地址與引用答對 | 引用存在 | 所引公告支持地址 |
| --- | --- | --- | --- | --- |
| 沒有公告 | 12/12 | 0/12 | 0/12 | 0/12 |
| 直接給正確公告 | 5/12 | 5/12 | 5/12 | 5/12 |
| 實際檢索取回 | 6/12 | 3/12 | 3/12 | 3/12 |
| 只有別家公告 | 7/12 | 0/12 | 5/12 | 5/12 |
| 正確公告加別家公告 | 2/12 | 2/12 | 12/12 | 2/12 |
| 同店公告只改地址 | 7/12 | 7/12 | 7/12 | 7/12 |
| 同店公告只改來源編號 | 5/12 | 5/12 | 5/12 | 5/12 |

檢索組找到九份公告，只有三題答出所問地址；另外三題沒找到而回答`UNKNOWN`，使協議成功數成為六。直接給正確公告仍錯七題，表示本輪不只有檢索瓶頸。地址與來源介入的兩版皆對分母都是完整12對，不能只挑原版答對的五題來報全體成功率。各題原始生成在報告的`samples`裡。訓練代價與固定資料上的平均預測代價，按有效目標token計算：把模型給每個真值token的負自然log機率加總，再除以目標數，計數例子見[7.12](https://birdhackor.github.io/tiny-perceptron-vlm/7.12.html)。本表的生成答對率則按12道完整問題計算。兩種分母不同，不能把按token計算的代價與整題答對率混成一個成績。

</details>
